# Pyannote Diarization testing - Speaker Diarization with Pyannote on VAST (cloud based, use for reference possibly?)
### <https://vast.ai/article/speaker-diarization-with-pyannote-on-vast>
### <https://github.com/pyannote/pyannote-audio>

In [ ]:
# TODO take a good look at later functions in this notebook and how they are calling each of the sub-function

# user configurable settings
audio_file = "sample_data/Heat_diner_scene.mp3"
path_to_ffmpeg = "ffmpeg"


In [ ]:
# setup
from pathlib import Path
import os, sys
from pydantic_settings import BaseSettings, SettingsConfigDict
import log_config # to override default and use loguru instead
log_config.setup_logging()
from loguru import logger

import logging

logging.basicConfig()

# turn on all the debugging logs, likely not needed, overkill to the max, but eh, gotta catch 'em all
logging.getLogger("faster_whisper").setLevel(logging.DEBUG)
logging.getLogger("whisper").setLevel(logging.DEBUG)
logging.getLogger("whisperx").setLevel(logging.DEBUG)
logging.getLogger("pyannote").setLevel(logging.DEBUG)

class NotebookSettings(BaseSettings):
    hf_token: str
    model_config = SettingsConfigDict(env_file=".env", env_file_encoding="utf-8")


settings = NotebookSettings()

# this hacky path trickery will unlikely to be needed in actual project, but I'm just trying to get this snippet to function in isolation to work for now, torchcodec BS has been bypassed thanks to ffmpeg's amazingly broad and powerful functionality to decode audio all by itself, with a myriad of various formats/codecs supported
# a bit of a hack to get windows to find the DLLs for FFmpeg
# Python ≥3.8 on Windows does not use PATH for dependent DLLs. You must add the FFmpeg DLL folder to the current process with os.add_dll_directory(...) before importing torchcodec. Adding PATH via os.system("set PATH=...") does not affect the running process. Order is also tricky if you add multiple directories. - See https://discuss.huggingface.co/t/cannot-load-torchcodec/169260/3
# Use Python's Windows DLL API (3.8+). Add the folder that holds avcodec/avformat/avutil DLLs. # already changed, added fresh cloned copy of ffmpeg repo (make sure it's the shared version with all the DLLs!) to project root as a fallback in case the user doesn't already have ffmpeg (probably not very likely to happen, so project root ffmpeg/bin is likely the primary)
# TorchCodec README + version matrix: https://github.com/pytorch/torchcodec  (docs)
# Torchaudio FFmpeg install notes on Windows: https://docs.pytorch.org/audio/main/installation.html  (install tips)

ffmpeg_dll_dir = Path(path_to_ffmpeg) / "bin"

assert ffmpeg_dll_dir.exists(), ffmpeg_dll_dir

abs_dll_path = os.path.abspath(str(ffmpeg_dll_dir))
print(f"Adding {abs_dll_path} to DLL search path for ffmpeg DLLs...")
os.add_dll_directory(abs_dll_path)  # Python 3.8+ DLL search

import torch, torchcodec, platform, subprocess

print(f"Python executable path: {sys.executable} (v{platform.python_version()})")
print(f"`torch` library version {torch.__version__}")
print(f"`torchcodec` version {torchcodec.__version__}")

ffmpeg_process = subprocess.run(["ffmpeg", "-version"], check=False, shell=True, capture_output= True)
print(ffmpeg_process.stdout)

# this version compatability between these libraries is a frustrating mess, opting to completely bypass torchcodec and using ffmpeg directly instead


Adding c:\Users\RobynPfeifer\code\STAT405_AudioNotetaker\ffmpeg\bin to DLL search path for ffmpeg DLLs...
Python executable path: c:\Users\RobynPfeifer\code\STAT405_AudioNotetaker\.venv\Scripts\python.exe (v3.14.2)
`torch` library version 2.10.0+cpu
`torchcodec` version 0.10.0
b'ffmpeg version N-122750-g6ee3e59ce2-20260215 Copyright (c) 2000-2026 the FFmpeg developers\r\nbuilt with gcc 15.2.0 (crosstool-NG 1.28.0.1_403899e)\r\nconfiguration: --prefix=/ffbuild/prefix --pkg-config-flags=--static --pkg-config=pkg-config --cross-prefix=x86_64-w64-mingw32- --arch=x86_64 --target-os=mingw32 --enable-gpl --enable-version3 --disable-debug --enable-shared --disable-static --disable-w32threads --enable-pthreads --enable-iconv --enable-zlib --enable-libxml2 --enable-libvmaf --enable-fontconfig --enable-libharfbuzz --enable-libfreetype --enable-libfribidi --enable-vulkan --enable-libshaderc --enable-libvorbis --disable-libxcb --disable-xlib --disable-libpulse --enable-opencl --enable-gmp --enable-

: 

## Introduction

This notebook demonstrates how to implement Speaker Diarization using the Pyannote Audio library on VAST.ai's cloud computing platform. Speaker Diarization is the process of partitioning an audio stream into segments according to the speaker identity, answering the question "who spoke when?"

### Why Speaker Diarization Matters

Speaker Diarization provides several key benefits for audio processing pipelines:

1. **Speaker Identification**: It identifies different speakers in a conversation, meeting, or any multi-speaker audio recording.

2. **Improved Transcription**: When combined with speech-to-text systems, diarization allows for speaker-attributed transcripts, making it clear who said what.

3. **Processing Efficiency**: By segmenting audio by speaker and removing non-speech portions, diarization can significantly reduce the computational load for downstream tasks like speech recognition, allowing these systems to process only relevant speech segments rather than the entire audio file.

4. **Audio Indexing**: Makes audio content searchable by speaker, allowing users to find all segments where a specific person speaks.


### What This Notebook Does

In this notebook, we will:
- Set up the Pyannote Audio Speaker Diarization pipeline
- Process audio files to detect different speakers and their speaking turns
- Calculate speaking time for each identified speaker
- Identify regions with overlapping speech
- Extract and save speaker-specific segments from the input audio
- Play and verify the diarization results

The output will be a collection of audio files separated by speaker, making them ready for further processing in speech-to-text pipelines or speaker-specific analysis.


## Choosing an Instance

For running the Pyannote Speaker Diarization model on VAST.ai, you'll need a relatively modest GPU setup. The pyannote/speaker-diarization-3.1 model runs in pure PyTorch and is designed to be efficient. Here are the recommended specifications:

- GPU: A low-end GPU like an RTX 3060 or 4060 would be sufficient.
- VRAM: 6-8GB of VRAM should be adequate as the Pyannote diarization pipeline is relatively efficient.
- RAM: 8-16GB system RAM is recommended for processing audio files.
- Storage: At least 10GB for the model, dependencies, and your audio files.
- CUDA: Make sure the instance has CUDA installed (version 11.0+ recommended).
- Python: Python 3.8+ with PyTorch installed.


## Install Dependencies

In [3]:
# unneeded cell, pyannote.audio, pydub and datasets are already installed
#uv add pyannote.audio
#uv add pydub
#uv add librosa # this is the package that barfs out the below error
#uv add datasets


A potential issue: The `librosa` library requires Python version >=3.6,<3.10, and we're currently running `3.14.2`, so this is a ... problem. That being said, I need to investigate the usage of `librosa`, and determine if it's a required package or if it is just used to play the audio files below.

In [ ]:
# ffmpeg already installed. Will (almost certainly) be bundled together with the compiled/containerized/whateverized project upon deployment.


## Set up your Huggingface Token

Here we set our huggingface token as `HF_TOKEN`. We need this to access the model.

Ensure that you have accepted the terms for https://huggingface.co/pyannote/speaker-diarization-3.1 and https://huggingface.co/pyannote/segmentation-3.0. This model is free to use, but you must accept their terms.

In [ ]:
# Argh, gated models...
# Make sure you've accepted the user conditions at:
# https://huggingface.co/pyannote/speaker-diarization-3.1
# https://huggingface.co/pyannote/segmentation-3.0

HF_TOKEN = settings.hf_token


## Download Test Data

We will use a sample file from the AMI Meeting Corpus dataset https://huggingface.co/datasets/diarizers-community/ami, which is a collection of 100 hours of meeting recordings.

This code efficiently pulls a few sample files from the dataset. If you want to download the entire dataset there are better methods - see the Huggingface API.

In [6]:
from datasets import load_dataset
import os
import soundfile as sf

# Create a directory to save the files
os.makedirs("ami_samples", exist_ok = True)

# Load the dataset with the correct split
dataset = load_dataset("diarizers-community/ami", "ihm", split = "train", streaming = True)


# load any number of samples
n_samples = 1
samples = list(dataset.take(n_samples))

for i, sample in enumerate(samples):

    audio = sample["audio"]
    audio_array = audio["array"]
    sampling_rate = audio["sampling_rate"]
    
    # Calculate duration in seconds
    duration = len(audio_array) / sampling_rate
    
    # Use soundfile to save the audio
    output_path = f"ami_samples/sample_{i}.wav"
    sf.write(output_path, audio_array, sampling_rate)
    
    print(f"Saved {output_path} - Speaker: {sample['speakers']} - Duration: {duration:.2f} seconds")

# sample output:
# Saved ami_samples/sample_0.wav - Speaker: ['MIE080', 'MIE080', 'MIE083', 'MIE080', 'MIE080', 'MIE083', 'MIO026', 'MIE080', 'MIE083', 'MIE080', 'MIO026', 'MIO026', 'MIO026', 'MIE083', 'MIE083', 'MIE029', 'MIO026', 'MIE029', 'MIO026', 'MIE083', 'MIE083', 'MIE080', 'MIE029', 'MIO026', 'MIE083', 'MIE029', 'MIE080', 'MIE080', 'MIE029', 'MIE080', 'MIE080', 'MIE029', 'MIE080', 'MIE029', 'MIE080', 'MIE029', 'MIE080', 'MIE029', 'MIO026', 'MIE029', 'MIE083', 'MIE080', 'MIE029', 'MIE080', 'MIO026', 'MIE029', 'MIE083', 'MIE029', 'MIE080', 'MIE080', 'MIE029', 'MIE083', 'MIE083', 'MIO026', 'MIE083', 'MIE029', 'MIE080', 'MIE080', 'MIE029', 'MIO026', 'MIE083', 'MIE083', 'MIE029', 'MIE080', 'MIO026', 'MIE080', 'MIE083', 'MIE029', 'MIE029', 'MIE083', 'MIE083', 'MIE080', 'MIE080', 'MIE080', 'MIE029', 'MIE080', 'MIO026', 'MIE080', 'MIE083', 'MIE029', 'MIE080', 'MIE029', 'MIE080', 'MIE080', 'MIE029', 'MIE029', 'MIE080', 'MIE029', 'MIE080', 'MIE083', 'MIE083', 'MIE080', 'MIE080', 'MIE083', 'MIE029', 'MIE080', 'MIE080', 'MIE029', 'MIE083', 'MIE029', 'MIE029', 'MIE080', 'MIE029', 'MIE080', 'MIE029', 'MIE080', 'MIE029', 'MIE080', 'MIE083', 'MIE029', 'MIE083', 'MIE029', 'MIE080', 'MIE029', 'MIE080', 'MIE029', 'MIE029', 'MIE080', 'MIE080', 'MIE029', 'MIE080', 'MIE029', 'MIE080', 'MIE080', 'MIE080', 'MIE080', 'MIE029', 'MIO026', 'MIE080', 'MIO026', 'MIE080', 'MIO026', 'MIO026', 'MIE029', 'MIO026', 'MIE029', 'MIE029', 'MIO026', 'MIE029', 'MIE080', 'MIO026', 'MIE029', 'MIE080', 'MIE080', 'MIE083', 'MIE029', 'MIE080', 'MIE080', 'MIO026', 'MIE080', 'MIE080', 'MIO026', 'MIO026', 'MIE029', 'MIE083', 'MIE080', 'MIO026', 'MIO026', 'MIE029', 'MIO026', 'MIE080', 'MIO026', 'MIE029', 'MIE080', 'MIE080', 'MIE029', 'MIE029', 'MIE080', 'MIE080', 'MIO026', 'MIO026', 'MIE080', 'MIE080', 'MIE080', 'MIE083', 'MIE029', 'MIE080', 'MIE083', 'MIE083', 'MIE029', 'MIE083', 'MIE083', 'MIE083', 'MIE080', 'MIE083', 'MIE080', 'MIE083', 'MIE029', 'MIE083', 'MIE080', 'MIE083', 'MIE083', 'MIE083', 'MIE080', 'MIE083', 'MIE029', 'MIE083', 'MIE083', 'MIE029', 'MIE080', 'MIE083', 'MIE080', 'MIE083', 'MIE080', 'MIE083', 'MIE080', 'MIE029', 'MIE080', 'MIE083', 'MIE080', 'MIE080', 'MIE080', 'MIE029', 'MIE029', 'MIE080', 'MIE083', 'MIE029', 'MIE080', 'MIE083', 'MIO026', 'MIE080', 'MIE080', 'MIE029', 'MIE029', 'MIE029', 'MIE029', 'MIO026', 'MIE029', 'MIE080', 'MIE080', 'MIE083', 'MIE083', 'MIE080', 'MIE029', 'MIE083', 'MIE080', 'MIE083', 'MIE029', 'MIE080', 'MIE080', 'MIE029', 'MIE080', 'MIE029', 'MIE080', 'MIE080', 'MIO026', 'MIE029', 'MIO026', 'MIE080', 'MIE029', 'MIE083', 'MIE083', 'MIE029', 'MIE083', 'MIE080', 'MIE080', 'MIE080', 'MIE080', 'MIE029', 'MIE029', 'MIE080', 'MIE083', 'MIE083', 'MIE083', 'MIE083', 'MIE029', 'MIE083', 'MIE029', 'MIE029', 'MIE083', 'MIO026', 'MIE083', 'MIE080', 'MIE083', 'MIE029', 'MIE029', 'MIE029', 'MIE029', 'MIE083', 'MIE080', 'MIE029', 'MIO026', 'MIE029', 'MIE083', 'MIE029', 'MIE029', 'MIE083', 'MIE080', 'MIE029', 'MIE080', 'MIE029', 'MIE083', 'MIO026', 'MIE080', 'MIE083', 'MIE080', 'MIO026', 'MIE029', 'MIE029', 'MIE080', 'MIO026', 'MIO026', 'MIE080', 'MIE083', 'MIE029', 'MIE080', 'MIE083', 'MIE029', 'MIE083', 'MIE029', 'MIE029', 'MIE080', 'MIE083', 'MIE080', 'MIE083', 'MIE029', 'MIE080', 'MIE029', 'MIE083', 'MIE080', 'MIE029', 'MIE080', 'MIE029', 'MIE083', 'MIE029', 'MIE080', 'MIE029', 'MIE083', 'MIE029', 'MIE080', 'MIE080', 'MIE080', 'MIO026', 'MIE083', 'MIE080', 'MIO026', 'MIE083', 'MIE080', 'MIE083', 'MIE080', 'MIE029', 'MIE080', 'MIE083', 'MIE080', 'MIE029', 'MIE080', 'MIE080', 'MIE029', 'MIE083', 'MIE029', 'MIE080', 'MIE029', 'MIE083', 'MIE080', 'MIE083', 'MIE080', 'MIE029', 'MIE029', 'MIE083', 'MIE080', 'MIE029', 'MIE083', 'MIO026', 'MIE080', 'MIE029', 'MIE080', 'MIE083', 'MIE080', 'MIE029', 'MIE080', 'MIE083', 'MIE083', 'MIE029', 'MIE029', 'MIE080', 'MIE080', 'MIE029', 'MIE080', 'MIO026', 'MIE029', 'MIE080', 'MIE080', 'MIE080', 'MIE029', 'MIE083', 'MIE080', 'MIE083', 'MIE080', 'MIE083', 'MIE029', 'MIE083', 'MIE080', 'MIE029', 'MIE080', 'MIE080', 'MIE083', 'MIE080', 'MIE029', 'MIE083', 'MIE080', 'MIE029', 'MIE083', 'MIE080', 'MIE029', 'MIE029', 'MIE080', 'MIE080', 'MIE029', 'MIE029', 'MIE029', 'MIE083', 'MIE080', 'MIE029', 'MIE083', 'MIE029', 'MIE080', 'MIE080', 'MIO026', 'MIE083', 'MIE029', 'MIE029', 'MIO026', 'MIE029', 'MIO026', 'MIE083', 'MIE083', 'MIE080', 'MIE083', 'MIO026', 'MIO026', 'MIE080', 'MIE083', 'MIE083', 'MIE080', 'MIE083', 'MIE080', 'MIE083', 'MIE083', 'MIE083', 'MIE029', 'MIE083', 'MIE080', 'MIE029', 'MIE080', 'MIE029', 'MIE080', 'MIO026', 'MIE080', 'MIO026', 'MIE083', 'MIE080', 'MIO026', 'MIE083', 'MIE083', 'MIE083'] - Duration: 2080.83 seconds


ModuleNotFoundError: No module named 'huggingface_hub'

## Speaker Diarization

First we set up the Speaker Diarization pipeline.

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from pyannote.audio import Pipeline

pipeline = Pipeline.from_pretrained( # this call causes the giant error below
        "pyannote/speaker-diarization-3.1",
        token = HF_TOKEN,
)

# Move pipeline to appropriate device
pipeline = pipeline.to(device)

# giant error from pipeline call on line 7:
# z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\pyannote\audio\core\io.py:47: UserWarning: 
# torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
# * use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
# * fix torchcodec installation. Error message was:

# Could not load libtorchcodec. Likely causes:
#           1. FFmpeg is not properly installed in your environment. We support
#              versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
#              for each of those versions. Errors for versions not installed on
#              your system are expected; only the error for your installed FFmpeg
#              version is relevant. On Windows, ensure you've installed the
#              "full-shared" version which ships DLLs.
#           2. The PyTorch version (2.10.0+cpu) is not compatible with
#              this version of TorchCodec. Refer to the version compatibility
#              table:
#              https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
#           3. Another runtime dependency; see exceptions below.

#         The following exceptions were raised as we tried to load libtorchcodec:
        
# [start of libtorchcodec loading traceback]
# FFmpeg version 8:
# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1442, in load_library
#     ctypes.CDLL(path)
#     ~~~~~~~~~~~^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 433, in __init__
#     self._handle = self._load_library(name, mode, handle, winmode)
#                    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 451, in _load_library
#     return _LoadLibrary(self._name, winmode)
# FileNotFoundError: Could not find module '\\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core8.dll' (or one of its dependencies). Try using the full path with constructor syntax.

# The above exception was the direct cause of the following exception:

# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\_core\ops.py", line 57, in load_torchcodec_shared_libraries
#     torch.ops.load_library(core_library_path)
#     ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1444, in load_library
#     raise OSError(f"Could not load this library: {path}") from e
# OSError: Could not load this library: \\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core8.dll

# FFmpeg version 7:
# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1442, in load_library
#     ctypes.CDLL(path)
#     ~~~~~~~~~~~^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 433, in __init__
#     self._handle = self._load_library(name, mode, handle, winmode)
#                    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 451, in _load_library
#     return _LoadLibrary(self._name, winmode)
# FileNotFoundError: Could not find module '\\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core7.dll' (or one of its dependencies). Try using the full path with constructor syntax.

# The above exception was the direct cause of the following exception:

# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\_core\ops.py", line 57, in load_torchcodec_shared_libraries
#     torch.ops.load_library(core_library_path)
#     ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1444, in load_library
#     raise OSError(f"Could not load this library: {path}") from e
# OSError: Could not load this library: \\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core7.dll

# FFmpeg version 6:
# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1442, in load_library
#     ctypes.CDLL(path)
#     ~~~~~~~~~~~^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 433, in __init__
#     self._handle = self._load_library(name, mode, handle, winmode)
#                    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 451, in _load_library
#     return _LoadLibrary(self._name, winmode)
# FileNotFoundError: Could not find module '\\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core6.dll' (or one of its dependencies). Try using the full path with constructor syntax.

# The above exception was the direct cause of the following exception:

# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\_core\ops.py", line 57, in load_torchcodec_shared_libraries
#     torch.ops.load_library(core_library_path)
#     ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1444, in load_library
#     raise OSError(f"Could not load this library: {path}") from e
# OSError: Could not load this library: \\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core6.dll

# FFmpeg version 5:
# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1442, in load_library
#     ctypes.CDLL(path)
#     ~~~~~~~~~~~^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 433, in __init__
#     self._handle = self._load_library(name, mode, handle, winmode)
#                    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 451, in _load_library
#     return _LoadLibrary(self._name, winmode)
# FileNotFoundError: Could not find module '\\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core5.dll' (or one of its dependencies). Try using the full path with constructor syntax.

# The above exception was the direct cause of the following exception:

# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\_core\ops.py", line 57, in load_torchcodec_shared_libraries
#     torch.ops.load_library(core_library_path)
#     ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1444, in load_library
#     raise OSError(f"Could not load this library: {path}") from e
# OSError: Could not load this library: \\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core5.dll

# FFmpeg version 4:
# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1442, in load_library
#     ctypes.CDLL(path)
#     ~~~~~~~~~~~^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 433, in __init__
#     self._handle = self._load_library(name, mode, handle, winmode)
#                    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 451, in _load_library
#     return _LoadLibrary(self._name, winmode)
# FileNotFoundError: Could not find module '\\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core4.dll' (or one of its dependencies). Try using the full path with constructor syntax.

# The above exception was the direct cause of the following exception:

# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\_core\ops.py", line 57, in load_torchcodec_shared_libraries
#     torch.ops.load_library(core_library_path)
#     ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1444, in load_library
#     raise OSError(f"Could not load this library: {path}") from e
# OSError: Could not load this library: \\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core4.dll
# [end of libtorchcodec loading traceback].
#   warnings.warn(

# it wasn't satisfied with just that giant dump of errors, so it appended one more, thankfully an easy fix...
# ---------------------------------------------------------------------------
# TypeError                                 Traceback (most recent call last)
# Cell In[7], line 7
#       3 device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#       5 from pyannote.audio import Pipeline
# ----> 7 pipeline = Pipeline.from_pretrained(
#       8         "pyannote/speaker-diarization-3.1",
#       9         use_auth_token = HF_TOKEN
#      10 )
#      12 # Move pipeline to appropriate device
#      13 pipeline = pipeline.to(device)

# TypeError: Pipeline.from_pretrained() got an unexpected keyword argument 'use_auth_token'

# k...

# changed named parameter from use_auth_token to token
# and it worked! Kinda!
# New output:
# config.yaml:   0%|          | 0.00/469 [00:00<?, ?B/s]
# pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]
# ---------------------------------------------------------------------------
# UnpicklingError                           Traceback (most recent call last)
# Cell In[9], line 7
#       3 device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#       5 from pyannote.audio import Pipeline
# ----> 7 pipeline = Pipeline.from_pretrained( # this call causes the giant error below
#       8         "pyannote/speaker-diarization-3.1",
#       9         token = HF_TOKEN,
#      10 )
#      12 # Move pipeline to appropriate device
#      13 pipeline = pipeline.to(device)

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\pyannote\audio\core\pipeline.py:244, in Pipeline.from_pretrained(cls, checkpoint, revision, hparams_file, token, cache_dir)
#     242 params.setdefault("token", token)
#     243 params.setdefault("cache_dir", cache_dir)
# --> 244 pipeline = Klass(**params)
#     246 # save pipeline origin (HF, local, etc) and class name as attributes for telemetry purposes
#     247 pipeline._otel_origin = otel_origin

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\pyannote\audio\pipelines\speaker_diarization.py:222, in SpeakerDiarization.__init__(self, legacy, segmentation, segmentation_step, embedding, embedding_exclude_overlap, plda, clustering, embedding_batch_size, segmentation_batch_size, der_variant, token, cache_dir)
#     219 self.legacy = legacy
#     221 self.segmentation_model = segmentation
# --> 222 model: Model = get_model(segmentation, token=token, cache_dir=cache_dir)
#     224 self.segmentation_step = segmentation_step
#     226 self.embedding = embedding

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\pyannote\audio\pipelines\utils\getter.py:115, in get_model(model, token, cache_dir)
#     112     pass
#     114 elif isinstance(model, str):
# --> 115     _model = Model.from_pretrained(
#     116         model,
#     117         token=token,
#     118         cache_dir=cache_dir,
#     119         strict=False,
#     120     )
#     121     if _model:
#     122         model = _model

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\pyannote\audio\core\model.py:602, in Model.from_pretrained(cls, checkpoint, map_location, strict, subfolder, revision, token, cache_dir, **kwargs)
#     599     map_location = default_map_location
#     601 # load checkpoint using lightning
# --> 602 loaded_checkpoint = pl_load(path_to_model_checkpoint, map_location=map_location)
#     604 # check that the checkpoint is compatible with the current version
#     605 versions = loaded_checkpoint["pyannote.audio"]["versions"]

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\lightning\fabric\utilities\cloud_io.py:73, in _load(path_or_url, map_location, weights_only)
#      71 fs = get_filesystem(path_or_url)
#      72 with fs.open(path_or_url, "rb") as f:
# ---> 73     return torch.load(
#      74         f,
#      75         map_location=map_location,  # type: ignore[arg-type]
#      76         weights_only=weights_only,
#      77     )

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\serialization.py:1548, in load(f, map_location, pickle_module, weights_only, mmap, **pickle_load_args)
#    1540                 return _load(
#    1541                     opened_zipfile,
#    1542                     map_location,
#    (...)   1545                     **pickle_load_args,
#    1546                 )
#    1547             except pickle.UnpicklingError as e:
# -> 1548                 raise pickle.UnpicklingError(_get_wo_message(str(e))) from None
#    1549         return _load(
#    1550             opened_zipfile,
#    1551             map_location,
#    (...)   1554             **pickle_load_args,
#    1555         )
#    1556 if mmap:

# UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
# 	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
# 	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
# 	WeightsUnpickler error: Unsupported global: GLOBAL torch.torch_version.TorchVersion was not an allowed global by default. Please use `torch.serialization.add_safe_globals([torch.torch_version.TorchVersion])` or the `torch.serialization.safe_globals([torch.torch_version.TorchVersion])` context manager to allowlist this global if you trust this class/function.

# Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.


: 

In [ ]:
# UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
# 	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
# 	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
# 	WeightsUnpickler error: Unsupported global: GLOBAL torch.torch_version.TorchVersion was not an allowed global by default. Please use `torch.serialization.add_safe_globals([torch.torch_version.TorchVersion])` or the `torch.serialization.safe_globals([torch.torch_version.TorchVersion])` context manager to allowlist this global if you trust this class/function.

# Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.


: 

### Get Diarization Results
Next, we process the file to get the timestamps where speech starts and ends.

In [ ]:
# Process the audio file
audio_file = "./ami_samples/sample_0.wav"
print(f"Processing {audio_file} on {device}")
output = pipeline(audio_file)


: 

The Pyannote Speaker Diarization model gives us a list of segment timestamps labeled with a speaker. 

In [ ]:
print(f"Voice activity segments:")

for segment, _, speaker in output.itertracks(yield_label = True):
        result = f"{segment.start:.2f} --> {segment.end:.2f} (duration: {segment.duration:.2f}s) Speaker: {speaker}"
        print(result)


: 

### Additional Analytics

Now that we have processed our file we'll explore a few useful features of the Pyannote SDK:

1. See total speaker time broken down by speaker.
2. Find segments with speaker overlap (multiple speakers speaking at once).
3. Filter the data by speaker.

#### Speaker Time

Here we see the total speaking time for each speaker.

In [ ]:
for speaker in output.labels():
    speaking_time = output.label_duration(speaker)
    print(f"Speaker {speaker} total speaking time: {speaking_time:.2f}s")


: 

#### Speaker Overlap

Pyannote shows us the timestamps where multiple speakers are speaking.

In [ ]:
overlap = output.get_overlap()
print(f"Overlapping speech regions: {overlap}")


: 

#### Filter the Data by Speaker

We can use Pyannote to filter the output by speaker.

In [ ]:
speaker = "SPEAKER_06"
speaker_turns = output.label_timeline(speaker)
print(f"Speaker {speaker} speaks at:")
for speaker_turn in speaker_turns:
    print(speaker_turn)


: 

## Inspect results

Next, we'll split the audio into chunks based on the diarization output in order to verify that it successfully isolated the speakers.


### Split the Audio

Here we write a function to split the original audio into segments determined by our Diarization output.

In [ ]:
import shutil
from pydub import AudioSegment

def split_audio_by_segments(audio_path, diarization_output, output_dir="output_segments"):
    """
    Split an audio file into multiple files based on diarization output
    
    Parameters:
    -----------
    audio_path: str
        Path to the input audio file
    diarization_output: Annotation
        Pyannote diarization output
    output_dir: str
        Directory to save the output segments
    """
    # Clear the output directory if it exists
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    
    # Create output directory
    os.makedirs(output_dir, exist_ok = True)
    
    # Load the audio file
    audio = AudioSegment.from_file(audio_path)
    
    # Extract each segment with speaker information
    for i, (segment, _, speaker) in enumerate(diarization_output.itertracks(yield_label = True)):
        # Convert seconds to milliseconds
        start_ms = int(segment.start * 1000)
        end_ms = int(segment.end * 1000)
        
        # Extract segment
        segment_audio = audio[start_ms:end_ms]
        
        # Generate output filename with speaker information
        filename = os.path.basename(audio_path)
        name, ext = os.path.splitext(filename)
        output_path = os.path.join(output_dir, f"{name}_segment_{i+1:04d}_{start_ms:08d}ms-{end_ms:08d}ms_speaker_{speaker}{ext}")
        
        # Export segment
        segment_audio.export(output_path, format = ext.replace('.', ''))
        print(f"Saved segment {i+1} to {output_path} (Speaker: {speaker})")


: 

We then use that to save the segments to a local folder

In [ ]:
split_audio_by_segments(audio_file, output)


: 

### Inspect Results

Here we create a function that allows us to play the audio files in our notebook.

In [ ]:
# import librosa
# from IPython.display import Audio, display

# def play_audio(file_path, sr = None):
#     """
#     Play an audio file in a Jupyter notebook.
    
#     Parameters:
#     -----------
#     file_path : str
#         Path to the audio file to play
#     sr : int, optional
#         Sample rate to load the audio with. If None, uses the file's native sample rate.
        
#     Returns:
#     --------
#     Audio widget that can be played in the notebook
    
#     Example:
#     --------
#     >>> play_audio('path/to/audio.wav')
#     """
#     # Load the audio file
#     y, sr = librosa.load(file_path, sr = sr)
    
    
#     # Display an audio widget to play the sound
#     audio_widget = Audio(data = y, rate = sr)
#     display(audio_widget)


import simpleaudio as sa

wave_obj = sa.WaveObject.from_wave_file(audio_file)
play_obj = wave_obj.play()
play_obj.wait_done()


: 

We'll use `play_audio` to listen to a few clips to verify that the speakers were correctly identified and isolated.

In [ ]:
import os
audio_dir = "./output_segments/"

audio_files = os.listdir(audio_dir)
audio_files.sort()

n_offset = 21
n_clips = 5

for fname in audio_files[n_offset:n_clips + n_offset]:
    print(f"File: {fname}")
    
    # Extract speaker info if present in filename
    if "_speaker_" in fname:
        speaker_part = fname.split("_speaker_")[1].split(".")[0]
        print(f"Speaker: {speaker_part}")
    
    #play_audio(audio_dir + fname)


: 

### Verify Speaker Overlap

Sometimes we find clips with multiple speakers speaking. We can check the overlap file to verify that there are multiple speakers speaking at that time. 

In [ ]:
overlap = output.get_overlap()
for overlap_ts in overlap:
    print(f"Overlapping speech regions: {overlap_ts}")


: 

## Conclusion

The Pyannote speaker diarization model successfully identified multiple distinct speakers in the AMI Meeting Corpus sample. The model accurately detected overlapping speech regions, which we confirmed through our audio extraction and playback tests, demonstrating its effectiveness at handling complex conversational dynamics.
